In [3]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('punkt_tab')

# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Dictionary of slang words and their replacements (English)
slang_dict = {
    "w": "good",
    "tbh": "to be honest",
    "omg": "oh my god",
    "lol": "laugh out loud",
    "idk": "i do not know",
    "brb": "be right back",
    "btw": "by the way",
    "imo": "in my opinion",
    "smh": "shaking my head",
    "fyi": "for your information",
    "np": "no problem",
    "ikr": "i know right",
    "asap": "as soon as possible",
    "bff": "best friend forever",
    "gg": "good game",
    "hmu": "hit me up",
    "dm": "direct message",
    "wtf": "what the fuck",
    "omw": "on my way",
    "tmi": "too much information",
    "jk": "just kidding",
    "rn": "right now",
    "pls": "please",
    "thx": "thanks",
    "u": "you",
    "ur": "your"
}

# Dictionary for Malay informal words
malay_dict = {
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "sbb": "sebab",
    "kt": "kata",
    "dkt": "dekat",
    "kat": "pada",
    "x": "tidak",
    "tk": "tidak",
    "tak": "tidak",
    "nk": "nak",
    "nak": "mahu",
    "pki": "pakai",
    "pke": "pakai",
    "sgt": "sangat",
    "sngt": "sangat",
    "dh": "sudah",
    "udh": "sudah",
    "da": "sudah",
    "dah": "sudah",
    "tpi": "tetapi",
    "tp": "tetapi",
    "klu": "kalau",
    "klo": "kalau",
    "kl": "kalau"
}

# Contractions dictionary
contractions_dict = {
    "wasn't": "was not",
    "isn't": "is not",
    "aren't": "are not",
    "weren't": "were not",
    "doesn't": "does not",
    "don't": "do not",
    "didn't": "did not",
    "can't": "cannot",
    "couldn't": "could not",
    "shouldn't": "should not",
    "wouldn't": "would not",
    "won't": "will not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "i'm": "i am",
    "you're": "you are",
    "he's": "he is",
    "she's": "she is",
    "it's": "it is",
    "we're": "we are",
    "they're": "they are",
    "i've": "i have",
    "you've": "you have",
    "we've": "we have",
    "they've": "they have",
    "i'd": "i would",
    "you'd": "you would",
    "he'd": "he would",
    "she'd": "she would",
    "we'd": "we would",
    "they'd": "they would",
    "i'll": "i will",
    "you'll": "you will",
    "he'll": "he will",
    "she'll": "she will",
    "we'll": "we will",
    "they'll": "they will",
    "let's": "let us",
    "that's": "that is",
    "who's": "who is",
    "what's": "what is",
    "where's": "where is",
    "when's": "when is",
    "why's": "why is",
    "there's": "there is",
    "here's": "here is"
}

# Common misspellings dictionary
misspellings_dict = {
    "enviroment": "environment",
    "managemn": "management",
    "opinjon": "opinion",
    "funtion": "function",
    "accomodation": "accommodation",
    "accomodations": "accommodations",
    "uniten": "UNITEN",
    "uniten": "UNITEN",
    "lecturer": "lecturer",
    "lecturers": "lecturers",
    "facilities": "facilities",
    "facility": "facility",
    "transportation": "transportation",
    "university": "university",
    "universities": "universities",
    "student": "student",
    "students": "students",
    "parking": "parking",
    "library": "library",
    "wifi": "wifi",
    "internet": "internet",
    "connection": "connection",
    "management": "management",
    "schedule": "schedule",
    "registration": "registration",
    "semester": "semester",
    "sem": "semester"
}

def clean_text(text):
    """Main function to clean and preprocess text"""
    if not isinstance(text, str):
        return ""
    
    # Handle special cases
    if text == "#NAME?":
        return ""
    
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove timestamps and special patterns
    text = re.sub(r'\d{1,2}:\d{2}:\d{2}\s*(am|pm)?', '', text)
    text = re.sub(r'gmt[+-]\d{1,2}', '', text)
    
    # Step 3: Remove special characters but keep letters and spaces
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # Step 4: Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Step 5: Replace contractions
    for contraction, expansion in contractions_dict.items():
        text = re.sub(r'\b' + contraction + r'\b', expansion, text)
    
    # Step 6: Replace English slang
    for slang, replacement in slang_dict.items():
        text = re.sub(r'\b' + slang + r'\b', replacement, text)
    
    # Step 7: Replace Malay informal words
    for informal, formal in malay_dict.items():
        text = re.sub(r'\b' + informal + r'\b', formal, text)
    
    # Step 8: Fix common misspellings
    for wrong, correct in misspellings_dict.items():
        text = re.sub(r'\b' + wrong + r'\b', correct, text)
    
    # Step 9: Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

def tokenize_and_lemmatize(text):
    """Tokenize and lemmatize the text"""
    if not isinstance(text, str) or text == "":
        return []
    
    try:
        # Tokenize
        tokens = word_tokenize(text)
        
        # Remove stopwords
        tokens = [token for token in tokens if token.lower() not in stop_words]
        
        # Lemmatize
        lemmatized = [lemmatizer.lemmatize(token) for token in tokens]
        
        return lemmatized
    except:
        return text.split()

def analyze_issues(df):
    """Analyze and report issues in the dataset"""
    print("=" * 60)
    print("ISSUES IDENTIFIED IN THE REVIEW COLUMN")
    print("=" * 60)
    
    # Check for missing values
    missing = df['Review'].isna().sum()
    print(f"\n1. Missing values: {missing} rows")
    
    # Check for empty strings
    empty = (df['Review'].astype(str).str.strip() == '').sum()
    print(f"2. Empty strings: {empty} rows")
    
    # Check for non-string values
    non_string = df['Review'].apply(lambda x: not isinstance(x, str)).sum()
    print(f"3. Non-string values: {non_string} rows")
    
    # Sample issues
    print("\n4. SAMPLE ISSUES FOUND:")
    print("-" * 40)
    
    # Check for contractions
    contraction_pattern = r"\b(" + "|".join(contractions_dict.keys()) + r")\b"
    with_contractions = df['Review'].astype(str).str.contains(contraction_pattern, case=False, na=False).sum()
    print(f"   - Contractions: {with_contractions} rows")
    
    # Check for slang
    slang_pattern = r"\b(" + "|".join(slang_dict.keys()) + r")\b"
    with_slang = df['Review'].astype(str).str.contains(slang_pattern, case=False, na=False).sum()
    print(f"   - Slang words: {with_slang} rows")
    
    # Check for Malay
    malay_pattern = r"\b(" + "|".join(malay_dict.keys()) + r")\b"
    with_malay = df['Review'].astype(str).str.contains(malay_pattern, case=False, na=False).sum()
    print(f"   - Malay informal words: {with_malay} rows")
    
    # Check for misspellings
    misspelling_pattern = r"\b(" + "|".join(misspellings_dict.keys()) + r")\b"
    with_misspellings = df['Review'].astype(str).str.contains(misspelling_pattern, case=False, na=False).sum()
    print(f"   - Common misspellings: {with_misspellings} rows")
    
    # Check for special characters
    special_chars = df['Review'].astype(str).str.contains(r'[^\w\s]', na=False).sum()
    print(f"   - Special characters: {special_chars} rows")
    
    # Check for numbers
    with_numbers = df['Review'].astype(str).str.contains(r'\d', na=False).sum()
    print(f"   - Contains numbers: {with_numbers} rows")
    
    # Check for timestamps
    with_timestamps = df['Review'].astype(str).str.contains(r'\d{1,2}:\d{2}', na=False).sum()
    print(f"   - Contains timestamps: {with_timestamps} rows")
    
    print("\n" + "=" * 60)

# Main execution
def main():
    # Load the dataset
    print("Loading UNITENReview.csv...")
    df = pd.read_csv("UNITENReview.csv")
    
    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    
    # Analyze issues
    analyze_issues(df)
    
    # Show sample of original reviews
    print("\nSAMPLE OF ORIGINAL REVIEWS:")
    print("-" * 40)
    for i, review in enumerate(df['Review'].head(3)):
        print(f"{i+1}. {review}")
    
    # Apply preprocessing
    print("\n" + "=" * 60)
    print("APPLYING TEXT PREPROCESSING...")
    print("=" * 60)
    
    # Clean the text
    df['cleaned_review'] = df['Review'].apply(clean_text)
    
    # Tokenize and lemmatize
    df['processed_tokens'] = df['cleaned_review'].apply(tokenize_and_lemmatize)
    
    # Join tokens back to text
    df['final_processed'] = df['processed_tokens'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
    
    # Show sample of processed reviews
    print("\nSAMPLE OF PROCESSED REVIEWS:")
    print("-" * 40)
    for i in range(min(3, len(df))):
        print(f"{i+1}. Original: {df['Review'].iloc[i]}")
        print(f"   Processed: {df['final_processed'].iloc[i]}")
        print()
    
    # Save the results
    output_file = "UNITENReview_Processed.csv"
    df.to_csv(output_file, index=False)
    print(f"\n✓ Processed data saved to: {output_file}")
    
    # Display preprocessing summary
    print("\n" + "=" * 60)
    print("PREPROCESSING SUMMARY")
    print("=" * 60)
    print(f"Total reviews processed: {len(df)}")
    print(f"Reviews with non-empty processed text: {(df['final_processed'].str.len() > 0).sum()}")
    print(f"Average token length after processing: {df['processed_tokens'].apply(len).mean():.2f}")

# Run the main function
if __name__ == "__main__":
    main()

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data

Loading UNITENReview.csv...

Dataset shape: (52, 2)
Columns: ['Timestamp', 'Review']
ISSUES IDENTIFIED IN THE REVIEW COLUMN

1. Missing values: 0 rows
2. Empty strings: 0 rows
3. Non-string values: 0 rows

4. SAMPLE ISSUES FOUND:
----------------------------------------
   - Contractions: 8 rows
   - Slang words: 3 rows
   - Malay informal words: 1 rows
   - Common misspellings: 45 rows
   - Special characters: 42 rows
   - Contains numbers: 4 rows
   - Contains timestamps: 0 rows


SAMPLE OF ORIGINAL REVIEWS:
----------------------------------------
1. Im happy with uniten actually, even the people are W
2. I’m having a pretty good time here, happy to meet all of the W people.
3. a very neutral place in terms of everything

APPLYING TEXT PREPROCESSING...

SAMPLE OF PROCESSED REVIEWS:
----------------------------------------
1. Original: Im happy with uniten actually, even the people are W
   Processed: im happy UNITEN actually even people good

2. Original: I’m having a pretty good ti